# 🧊 11 · 3D 视觉：点云与 NeRF 与 3DGS

> 三范式一条线：**显式点云**（离散几何）→ **隐式 NeRF**（连续神经场）→ **显式可微 3DGS**（高斯泼溅）。

为什么从「给 2D 图像」走向「重建 3D 世界」？——自动驾驶/机器人/数字人/AIGC 都需要**几何 + 视角**，本节讲清三范式核心与手撕代码。

## 📋 本节要解决的问题

| # | 问题 | 范式 |
|---|---|---|
| 1 | 3D 数据长什么样？ | 点云：无序集合 |
| 2 | 无序数据怎么进网络？ | PointNet 对称聚合 |
| 3 | 怎么从 2D 图重建 3D？ | NeRF 神经辐射场 |
| 4 | 体渲染公式？ | alpha compositing |
| 5 | 重建太慢怎么办？ | 3DGS 显式高斯 + 光栅化 |
| 6 | 三范式怎么选？ | 速度/质量/可控性对比 |

## 1️⃣ 点云：最原始的显式 3D

- **来源**：LiDAR（自动驾驶）、深度相机、多视图三角化、SFM/MVS；
- **格式**：$N$ 个点 $(x,y,z)$（可带强度/颜色/法线）；**无序**、**稀疏**、**密度不均**（近密远疏）；
- 基本操作：体素/最远点下采样、法线估计（局部 PCA）、配准（ICP）、分割/检测；
- 挑战：无序性（点序置换不变）、稀疏性、大点数（百万级）的实时处理。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(0)

rng = np.random.RandomState(0)
phi = rng.uniform(0, 2*np.pi, 600)
cos = rng.uniform(-1, 1, 600)                          # 球面均匀采样
pts = np.stack([np.sqrt(1-cos**2)*np.cos(phi),
                np.sqrt(1-cos**2)*np.sin(phi), cos], axis=1)*3
pts += rng.normal(0, 0.05, pts.shape)
fig = plt.figure(figsize=(5.5, 4.2))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*pts.T, s=2, c=pts[:, 2], cmap='viridis')
ax.set_title('球面点云：表面离散采样（LiDAR 类比）')
plt.show(); print('点云形状 (N, 3) =', pts.shape)

In [ ]:
def voxel_downsample(pts, size):
    keys = np.floor(pts/size).astype(int)
    _, idx = np.unique(keys, axis=0, return_index=True)
    return pts[idx]

def estimate_normals(pts, k=8):
    norms = []
    for i in range(len(pts)):
        d = np.sum((pts - pts[i])**2, axis=1)
        idx = np.argpartition(d, k)[:k]          # 手写 KNN（小样本够用）
        w, v = np.linalg.eigh(np.cov(pts[idx].T))
        norms.append(v[:, 0])                    # 最小特征向量 = 法线
    return np.array(norms)

down = voxel_downsample(pts, 0.8)
normals = estimate_normals(down)
print('降采样: %d → %d 点（体素大小 0.8）' % (len(pts), len(down)))
print('法线长度应=1: %.6f' % np.linalg.norm(normals[0]))
print('✓ 体素下采样 + PCA 法线估计（点云预处理三件套之二）')

## 2️⃣ PointNet：点云深度学习入口

点云无序 → 网络必须**置换不变**。PointNet 的答案：

$$ f(x_1,\dots,x_N) = \gamma\left( \max_{i} \, h(x_i) \right) $$

- 每个点共享 MLP $h$（1×1 卷积等价于逐点 MLP），**max-pool 聚合**全局特征 $\gamma$；
- max 是对称函数 → **对点序完全不变**（这就是灵魂）；
- 局限：**没有局部结构**（点与点关系只靠 max），对细粒度分割/遮挡鲁棒性弱；
- T-Net 对齐（输入/特征空间旋转归一化）可选组件。

In [ ]:
def pointnet(x, W1, b1, W2, b2):
    h = np.maximum(0, x @ W1 + b1)
    h = np.maximum(0, h @ W2 + b2)
    return h.max(axis=0)                     # max-pool 全局特征

rng = np.random.RandomState(1)
N, D, H = 1024, 3, 32
x = rng.randn(N, D)
W1 = rng.randn(D, H)*0.1; b1 = rng.randn(H)*0.01
W2 = rng.randn(H, H)*0.1; b2 = rng.randn(H)*0.01
g1 = pointnet(x, W1, b1, W2, b2)
perm = rng.permutation(N)
g2 = pointnet(x[perm], W1, b1, W2, b2)      # 打乱点序
print('打乱点序后全局特征最大差 = %.2e' % np.abs(g1-g2).max())
assert np.abs(g1-g2).max() < 1e-9
print('✓ PointNet max-pool 置换不变性（对称性只针对点序）')

## 3️⃣ PointNet++：引入局部结构

PointNet 缺局部——PointNet++ 用**层级分组**：

- **FPS 最远点采样**挑中心点（覆盖均匀）；
- 以中心点做**球邻域分组**（或 KNN），组内 PointNet 提取局部特征；
- 逐层抽象：局部 → 更大范围 → 全局，类似 CNN 感受野；
- 分割需要**上采样插值**（feature propagation）把点级特征传回。

后续工作：稀疏卷积（MinkowskiNet）、Transformer 点云（Point Transformer）延续「局部+全局」范式。

## 4️⃣ NeRF：隐式神经辐射场（2020 ECCV Best Paper）

目标：给多张不同视角的照片，重建任意新视角。

- 网络输入 **5D 坐标** $(x,y,z,\theta,\phi)$ → 输出**颜色 c** 与**密度 σ**；
- 对每条视线 ray 采样 N 个点，**体渲染**累积成像素颜色；
- 训练：预测像素 vs 真实像素的 L2，仅需多视图图像（自监督）；
- **位置编码** $\gamma(p)=(\sin 2^l\pi p, \cos 2^l\pi p)$ 把低频坐标映射到高频基——否则 MLP 学不出细节（谱偏置）。

## 5️⃣ 体渲染公式（alpha compositing）

沿射线采样 $N$ 点，间隔 $\delta_i$，密度 $\sigma_i$：

$$ \hat C = \sum_{i=1}^{N} T_i\,\alpha_i\, c_i, \qquad \alpha_i = 1-e^{-\sigma_i \delta_i}, \qquad T_i = e^{-\sum_{j<i}\sigma_j \delta_j} $$

- $\alpha_i$：该点遮挡比例（不透度）；$T_i$：到达该点的**透射率**（前面没被挡住的概率）；
- 前方物体挡后方：$T$ 随深度指数衰减——**近处密度大 → 远处看不到**；
- 3DGS 渲染用**同一公式**（逐高斯合成）——三范式共享的数学底座。

In [ ]:
def render_ray(colors, sigma, delta):
    alpha = 1 - np.exp(-sigma*delta)
    T = np.empty_like(sigma); T[0] = 1.0
    for i in range(1, len(sigma)):
        T[i] = T[i-1]*np.exp(-sigma[i-1]*delta)
    return (T*alpha) @ colors, T, alpha

np.random.seed(2)
N = 40; delta = 0.05
sigma = np.zeros(N)
sigma[10:14] = 8.0                        # 前景物体（不透）
sigma[30:33] = 8.0                        # 背景物体（应被遮挡）
colors = np.stack([np.zeros(N), np.zeros(N), np.ones(N)], axis=1)*0.2
colors[10:14] = [1.0, 0.2, 0.2]           # 前景红
colors[30:33] = [0.2, 1.0, 0.2]           # 背景绿
C, T, alpha = render_ray(colors, sigma, delta)
print('渲染颜色 =', np.round(C, 3), '（应为偏红：背景被前景遮挡）')
print('到达背景前透射率 T = %.3f' % T[30])
assert C[0] > C[1] and C[2] < 0.05
print('✓ 体渲染正确实现遮挡（alpha compositing）')

In [ ]:
def positional_encoding(x, L):
    out = [x]
    for l in range(L):
        out += [np.sin(2**l*np.pi*x), np.cos(2**l*np.pi*x)]
    return np.concatenate(out)

x = np.linspace(-1, 1, 300)
pe = positional_encoding(x, L=5)
print('输入维度 1 → 编码维度 %d（2L+1）' % len(pe))
plt.figure(figsize=(7.5, 3.4))
for i in range(1, 6):
    plt.plot(x, np.sin(2**i*np.pi*x), lw=1.2, label=f'频率 2^{i}')
plt.xlabel('坐标 x'); plt.ylabel('基函数'); plt.legend(ncol=5, fontsize=8)
plt.title('位置编码：把平滑坐标映射到多频基，MLP 才能拟合高频细节')
plt.tight_layout(); plt.show()

## 6️⃣ 3DGS：3D Gaussian Splatting（2023 SIGGRAPH）

NeRF 每像素要 query 整条射线（慢）；3DGS 直接优化**显式的 3D 高斯**：

- 场景 = 几千~百万个 3D 高斯：位置 μ、**各向异性协方差 Σ**、颜色（球谐 SH）、不透明度；
- **可微光栅化**：高斯投影到 2D（仿射近似）→ 按深度排序 → alpha 合成；
- 训练几十秒~分钟（vs NeRF 小时~天），渲染实时 100+ FPS，质量相当——**新范式事实标准**。

## 7️⃣ 3DGS 关键公式

- 3D 高斯：$G(x) = e^{-\frac{1}{2}(x-\mu)^\top \Sigma^{-1}(x-\mu)}$，$\Sigma = RSS^\top R^\top$（旋转 $R$ × 缩放对角 $S$，保证半正定）；
- 投影：$\Sigma' = JW \Sigma W^\top J^\top$（视图变换 $W$ + 相机投影的仿射雅可比 $J$）；
- 每像素：$C = \sum_i T_i\, \alpha_i\, c_i$（**和 NeRF 完全相同的体渲染**）；
- 排序：按深度对高斯排序，**每 tile 独立并行**光栅化；
- 自适应控制：密度过大的地方**分裂/克隆**高斯，透明处**剪除**。

In [ ]:
def gauss2d(X, Y, mu, cov):
    inv = np.linalg.inv(cov); det = np.linalg.det(cov)
    d = np.stack([X-mu[0], Y-mu[1]], axis=-1)
    return np.exp(-0.5*np.einsum('...i,ij,...j->...', d, inv, d))
    
N = 64
X, Y = np.meshgrid(np.linspace(-3, 3, N), np.linspace(-3, 3, N))
gs = [
    {'mu': [-1.0, 0.0], 'cov': [[0.3, 0.1], [0.1, 0.2]]},
    {'mu': [0.8, 0.5],  'cov': [[0.5, 0.0], [0.0, 0.05]]},
    {'mu': [0.2, -1.0], 'cov': [[0.05, 0.0], [0.0, 0.6]]},
]
img = sum(gauss2d(X, Y, g['mu'], g['cov']) for g in gs)
plt.figure(figsize=(5.5, 4.5))
plt.imshow(img, origin='lower', cmap='hot')
plt.title('2D 高斯泼溅：各向异性椭圆按深度合成 = 场景')
plt.colorbar(); plt.tight_layout(); plt.show()
print('✓ 高斯形状由协方差 Σ 决定：旋转+缩放（3DGS 的最小渲染单元）')

## ⚖️ NeRF vs 3DGS

| 维度 | NeRF | 3DGS |
|---|---|---|
| 表示 | 隐式连续场 | 显式离散高斯 |
| 训练 | 数小时~天 | 分钟级 |
| 渲染 | 逐射线采样，慢 | 光栅化，实时 100+ FPS |
| 显存 | 网络小、但缓存多 | 高斯多，显存较高 |
| 可控性 | 密度/几何难编辑 | 高斯可增删/动画 |
| 应用 | 学术重建/新视角 | 数字人/自动驾驶/AIGC 工业主力 |

选型：要**实时 + 可编辑**选 3DGS；要**严谨几何场/体积物理**选 NeRF 系（Instant-NGP、NeuS 等）。

## 🎯 面试高频考点

1. 点云三大挑战（无序/稀疏/不均）与预处理（体素/FPS/法线/ICP）；
2. PointNet 置换不变性来源（对称 max）；PointNet++ 局部层级；
3. NeRF 5D 输入输出、体渲染公式、为什么需要位置编码；
4. alpha compositing 与透射率 T 的遮挡物理；
5. 3DGS 的高斯参数（μ, Σ=RSSᵀRᵀ, SH, α）与投影公式；
6. 3DGS vs NeRF 的速度/质量/可控性差异；
7. 位置编码的多频基直觉（谱偏置）；
8. 3D 视觉的应用落点：自动驾驶（LiDAR）、数字人、AR/VR、AIGC 3D 生成。

## ✅ 自测清单

- [ ] 手写体素下采样 + PCA 法线；
- [ ] 手写 PointNet 并验证置换不变；
- [ ] 手写体渲染 ray compositing 并验证遮挡；
- [ ] 手写位置编码并讲清多频基动机；
- [ ] 手绘 2D 高斯泼溅叠加；
- [ ] 说清 NeRF vs 3DGS 对比表全部条目；
- [ ] 复述 8 条考点各 30 秒。

**衔接**：3D 视觉与 04-目标检测/05-分割（BEV 视角）组合成自动驾驶感知栈；与 LLM 多模态（视频/世界模型）互渗——建议配 04-LLM 模块并行复习。

In [ ]:
# 手写 ICP：最近邻对应 + SVD 刚性变换，迭代配准
def icp(src, dst, iters=20):
    src_t = src.copy()
    for _ in range(iters):
        d2 = np.sum((src_t[:, None, :] - dst[None, :, :])**2, axis=2)
        idx = np.argmin(d2, axis=1)
        A, B = src_t, dst[idx]
        muA, muB = A.mean(axis=0), B.mean(axis=0)
        H = (A-muA).T @ (B-muB)
        U, _, Vt = np.linalg.svd(H)
        if np.linalg.det(U @ Vt) < 0: U[:, -1] *= -1     # 防反射
        R = U @ Vt
        src_t = (src_t - muA) @ R.T + muB
    return src_t

rng = np.random.RandomState(0)
src = rng.randn(60, 3)*0.5
th = 0.4
R = np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]])
dst = (R @ src.T).T + np.array([1.0, 0.5, 0.0])
def align_err(A, B):
    return np.mean(np.min(np.sum((A[:, None, :]-B[None, :, :])**2, axis=2), axis=1))
print('配准前误差 = %.4f | ICP 20 次后 = %.4f' % (align_err(src, dst), align_err(icp(src, dst), dst)))
assert align_err(icp(src, dst), dst) < align_err(src, dst)
print('✓ 手写 ICP：迭代最近点 + SVD 求刚体变换')

## 1️⃣0️⃣ 点云检测与分割

- **VoxelNet / PointPillars**：体素化（柱体化）→ 3D CNN / 2D CNN → 3D 框回归（自动驾驶检测主流）；
- **PointNet++ / 稀疏卷积**：分割（道路/车辆/行人）；
- 融合流派：**BEV**（鸟瞰图）+ 图像（Camera-LiDAR 融合，如 BEVFusion）；
- 指标：3D IoU、mAP；挑战：远距稀疏、小物体、雨雾。

In [ ]:
# 两视图三角化（DLT 奇异值分解）
def project(P_, R, t, f=500):
    X = R @ P_ + t
    return np.array([f*X[0]/X[2], f*X[1]/X[2]])

def triangulate(p1, p2, R1, t1, R2, t2):
    A = []
    for p, R, t in [(p1, R1, t1), (p2, R2, t2)]:
        M = np.column_stack([R, t.reshape(-1, 1)])
        A.append(p[1]*M[2] - M[1])          # y·P³ − P²
        A.append(M[0] - p[0]*M[2])          # P¹ − x·P³
    _, _, Vt = np.linalg.svd(np.array(A))
    X = Vt[-1]
    return X[:3]/X[3]

rng = np.random.RandomState(1)
P = np.array([5.0, 1.0, 3.0])
R2 = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]])
t2 = np.array([0.0, 2.0, 0.0])
X = triangulate(project(P, np.eye(3), np.zeros(3)), project(P, R2, t2),
                np.eye(3), np.zeros(3), R2, t2)
print('三角化点 =', np.round(X, 2), '| 真值 =', P)
assert np.allclose(X, P, atol=1e-2)
print('✓ 两视图三角化：多视图几何的起点（SfM/MVS 基石）')

## 1️⃣1️⃣ NeRF 加速与变体

- **Instant-NGP**：多分辨率**哈希网格** + 小 MLP → 训练从小时级降到秒-分钟级；
- **Mip-NeRF**：锥体采样替代射线采样，抗锯齿、跨尺度一致；
- **NeuS / VolSDF**：用 SDF 表示几何 → 可提取网格、更严几何；
- 共同点：把「慢」拆在（哈希/网格/位置编码）加速特征 + 小网络。

In [ ]:
# 密度场沿射线：前景遮挡后景的可视化
def render_props(sigma, delta):
    alpha = 1 - np.exp(-sigma*delta)
    T = np.empty_like(sigma); T[0] = 1.0
    for i in range(1, len(sigma)):
        T[i] = T[i-1]*np.exp(-sigma[i-1]*delta)
    return T, alpha

t_ray = np.linspace(0, 1, 80)
sigma = 6*np.exp(-((t_ray-0.3)**2)/0.003) + 6*np.exp(-((t_ray-0.7)**2)/0.003)
T, alpha = render_props(sigma, t_ray[1]-t_ray[0])
plt.figure(figsize=(7.5, 3.4))
plt.plot(t_ray, sigma, lw=2, label='密度 σ')
plt.plot(t_ray, T, lw=2, label='透射率 T')
plt.plot(t_ray, alpha, lw=2, label='不透明度 α')
plt.xlabel('射线深度'); plt.legend(); plt.grid(alpha=0.3)
plt.title('体渲染：前景处 T 骤降 → 后景几乎不可见')
plt.tight_layout(); plt.show()
print('后景（t=0.7）处 T = %.3f（被前景遮挡）' % T[int(0.7*80)])

## 1️⃣2️⃣ NeRF 分层采样（coarse → fine）

- 朴素均匀采样浪费：大部分空间是空的；
- **coarse 网络**：均匀采样 N 点算权重 $w_i = T_i\alpha_i$；
- **fine 网络**：按权重分布重采样 N 点（集中在有内容处）→ 两网络输出合成；
- 效果：同等样本数下细节大幅提升。

In [ ]:
rng = np.random.RandomState(2)
t = np.linspace(0, 1, 16)
sigma = 8*np.exp(-((t-0.4)**2)/0.005)            # 内容集中在 t≈0.4
w = 1 - np.exp(-sigma*(t[1]-t[0])); w = w/w.sum()
cdf = np.cumsum(w)
t_fine = t[np.clip(np.searchsorted(cdf, rng.rand(32)), 0, len(t)-1)]
plt.figure(figsize=(7.5, 3.4))
plt.hist(t_fine, bins=16, density=True, alpha=0.6, label='fine 采样分布')
plt.plot(t, sigma/sigma.max(), 'r-', lw=2, label='密度（归一化）')
plt.xlabel('射线深度'); plt.legend(); plt.grid(alpha=0.3)
plt.title('分层采样：fine 点集中在高密度区域')
plt.tight_layout(); plt.show()
print('✓ 粗到细：按权重重采样（NeRF 两阶段结构）')

## 1️⃣3️⃣ 3DGS 渲染管线细节

1. 高斯投影到屏幕（协方差经视图变换）；
2. 按 tile（16×16 像素块）分配高斯 → **tile 内深度排序**；
3. 逐 tile 并行 alpha 合成（与 NeRF 同一公式）；
4. 自适应：大/高不透明高斯分裂、小/透明剪除、梯度驱动移动；

训练 1~3 分钟 / 渲染 100+ FPS 的秘密：**排序在 tile 粒度** + **光栅化全并行**。

In [ ]:
# 3DGS 自适应控制：分裂大高斯、剪除透明高斯
rng = np.random.RandomState(3)
n = 100
sizes = rng.exponential(0.4, n)        # 部分高斯协方差过大
opacity = rng.uniform(0, 0.1, n)       # 部分几乎透明
n_split = int((sizes > 0.5).sum())
n_prune = int((opacity < 0.01).sum())
print('初始 %d 个高斯 → 分裂 %d 个、剪除 %d 个 → 预计 %d 个'
      % (n, n_split, n_prune, n + n_split - n_prune))
print('✓ 3DGS 密度自适应：几何不够细→分裂，透明冗余→剪除')

## 🎯 进阶追问集（3D 视觉）

- 「点云为什么无序？」→ 采集无固定顺序；置换不变性靠对称聚合（max pool）；
- 「NeRF 训练要什么？」→ 多视图图片 + 相机位姿（SfM 提供）；自监督重建；
- 「体渲染和传统渲染区别？」→ 可微 alpha 合成 vs 光栅化/光线追踪的显式几何；
- 「3DGS 为什么快？」→ 显式高斯免去每像素积分；tile 级排序并行；
- 「3D 生成（AIGC）？」→ 3DGS 作表示 + 扩散/VAE 生成（DreamGaussian、GaussianDreamer）；
- 「和世界模型关系？」→ 3D 表示是视频/世界模型的可控中间层（Geometry-aware）。

In [ ]:
# 点云法线估计：PCA 最小特征向量（面试常考手撕）
rng = np.random.RandomState(4)
n = 200
x = np.concatenate([np.linspace(-2, 2, n)[:, None], np.zeros((n, 1))])   # 平面 z=0
pts = x + rng.normal(0, 0.02, (n, 2))
pts = np.column_stack([pts, np.zeros(n)])
Xc = pts - pts.mean(axis=0)
cov = Xc.T @ Xc/n
w, V = np.linalg.eigh(cov)
normal = V[:, 0]                                # 最小特征向量
print('估计法线 =', np.round(normal, 4), '| 真值 = [0, 0, 1]')
print('✓ PCA 手撕法线：平坦面法线即最小方差方向（NeRF/SDF 也用它做几何正则）')

In [ ]:
# 体渲染合成颜色：沿射线累积 RGB（同 cv11-render 但逐色通道）
t = np.linspace(0, 1, 64)
rng = np.random.RandomState(5)
color = np.stack([rng.uniform(0.2, 0.9, len(t)) for _ in range(3)], axis=1)
sigma = 5*np.exp(-((t-0.45)**2)/0.002)          # 前表面
delta = t[1]-t[0]
alpha = 1 - np.exp(-sigma*delta)
T = np.empty(len(t)); T[0] = 1.0
for i in range(1, len(t)):
    T[i] = T[i-1]*np.exp(-sigma[i-1]*delta)
rgb = (T[:, None]*alpha[:, None]*color).sum(axis=0)
print('合成像素 RGB =', np.round(rgb, 3), '（表面越近 T 越大 → 颜色越主导）')
print('✓ 体渲染公式 C=Σ T·α·c：NeRF/3DGS 共用的可微合成')

## 🎯 3D 视觉场景题（面试）

- 「自动驾驶 3D 检测方案？」→ LiDAR BEV（PointPillars/VoxelNet）+ 相机 BEV 融合 → NMS → 轨迹；
- 「做数字人/重建？」→ 多视角采集 → COLMAP(SfM) → NeRF/3DGS → 实时渲染；
- 「SLAM 漂移怎么办？」→ 回环检测 + 图优化（BA）；单目加尺度约束；
- 「点云噪声？」→ 统计滤波（邻域距离）+ 体素下采样 + 法线一致性；
- 「3DGS 显存爆了？」→ 高斯数量控制（剪除/合并）、16bit 属性、分块 LOD。

## 🏁 三范式总结 + 复习路径

| 范式 | 表示 | 渲染 | 训练 | 场景 |
|---|---|---|---|---|
| 传统 | 点云/网格 | 光栅化 | 几何处理 | 配准/检测 |
| NeRF | 隐式 MLP(σ,c) | 体渲染 | 光度重建 | 新视角合成 |
| 3DGS | 显式高斯 | tile 光栅化 | 梯度自适应 | 实时渲染 |

复习路径：`python tools/exec_nbs.py 09-计算机视觉` 后按三范式各讲一遍「表示-渲染-训练-应用」。